# Decision Trees and Random Forests Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Gini impurity and entropy

Build both split criteria from scratch and verify they agree on which splits are good.

In [ ]:
```python

import math

def gini_impurity(labels):

    n = len(labels)

    if n == 0:

        return 0.0

    counts = {}

    for label in labels:

        counts[label] = counts.get(label, 0) + 1

    return 1.0 - sum((c / n) ** 2 for c in counts.values())

def entropy(labels):

    n = len(labels)

    if n == 0:

        return 0.0

    counts = {}

    for label in labels:

        counts[label] = counts.get(label, 0) + 1

    return -sum(

        (c / n) * math.log2(c / n) for c in counts.values() if c > 0

    )

In [ ]:
```

### Step 2: Find the best split

Try every feature and every threshold. Return the one with the highest information gain.

In [ ]:
```python

def information_gain(parent_labels, left_labels, right_labels, criterion="gini"):

    measure = gini_impurity if criterion == "gini" else entropy

    n = len(parent_labels)

    n_left = len(left_labels)

    n_right = len(right_labels)

    if n_left == 0 or n_right == 0:

        return 0.0

    parent_impurity = measure(parent_labels)

    child_impurity = (

        (n_left / n) * measure(left_labels) +

        (n_right / n) * measure(right_labels)

    )

    return parent_impurity - child_impurity

In [ ]:
```

### Step 3: Build the DecisionTree class

Recursive splitting, prediction, and feature importance tracking.

In [ ]:
```python

class DecisionTree:

    def __init__(self, max_depth=None, min_samples_split=2,

                 min_samples_leaf=1, criterion="gini",

                 max_features=None):

        self.max_depth = max_depth

        self.min_samples_split = min_samples_split

        self.min_samples_leaf = min_samples_leaf

        self.criterion = criterion

        self.max_features = max_features

        self.tree = None

        self.feature_importances_ = None

    def fit(self, X, y):

        self.n_features = len(X[0])

        self.feature_importances_ = [0.0] * self.n_features

        self.n_samples = len(X)

        self.tree = self._build(X, y, depth=0)

        total = sum(self.feature_importances_)

        if total > 0:

            self.feature_importances_ = [

                fi / total for fi in self.feature_importances_

            ]

    def predict(self, X):

        return [self._predict_one(x, self.tree) for x in X]

In [ ]:
```

### Step 4: Build the RandomForest class

Bootstrap sampling, feature randomization, and majority voting.

In [ ]:
```python

class RandomForest:

    def __init__(self, n_trees=100, max_depth=None,

                 min_samples_split=2, max_features="sqrt",

                 criterion="gini"):

        self.n_trees = n_trees

        self.max_depth = max_depth

        self.min_samples_split = min_samples_split

        self.max_features = max_features

        self.criterion = criterion

        self.trees = []

    def fit(self, X, y):

        n = len(X)

        for _ in range(self.n_trees):

            indices = [random.randint(0, n - 1) for _ in range(n)]

            X_boot = [X[i] for i in indices]

            y_boot = [y[i] for i in indices]

            tree = DecisionTree(

                max_depth=self.max_depth,

                min_samples_split=self.min_samples_split,

                max_features=self.max_features,

                criterion=self.criterion,

            )

            tree.fit(X_boot, y_boot)

            self.trees.append(tree)

    def predict(self, X):

        all_preds = [tree.predict(X) for tree in self.trees]

        predictions = []

        for i in range(len(X)):

            votes = {}

            for preds in all_preds:

                v = preds[i]

                votes[v] = votes.get(v, 0) + 1

            predictions.append(max(votes, key=votes.get))

        return predictions

In [ ]:
```

See `code/trees.py` for the complete implementation with all helper methods.

## Exercises

In [ ]:
1. Train a single decision tree on a 2D dataset with 3 classes. Manually trace the splits and draw the rectangular decision boundaries. Compare the boundaries at max_depth=2 vs max_depth=10.

2. Implement variance reduction splitting for regression trees. Generate y = sin(x) + noise for 200 points and fit your regression tree. Plot the tree's piecewise-constant predictions against the true curve.

3. Build a random forest with 1, 5, 10, 50, and 200 trees. Plot training accuracy and test accuracy vs number of trees. Observe that test accuracy plateaus but does not decrease (forests resist overfitting).

4. Compare Gini impurity vs entropy as split criteria on 5 different datasets. Measure accuracy and tree depth. In most cases, they produce nearly identical results. Explain why.

5. Implement permutation importance. Compare it with MDI importance on a dataset where one feature is random noise but has high cardinality. MDI will rank the noise feature highly. Permutation importance will not.